In [8]:
from google.colab import drive

drive.mount('/content/drive')

import sys
import os

PROJECT_PATH = '/content/drive/MyDrive/PycharmProjects/Quantum-Learn'
sys.path.append(PROJECT_PATH)

os.chdir(PROJECT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -r requirements.txt
!pip install --upgrade --force-reinstall "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install --upgrade jax-cuda12-plugin jax-cuda12-pjrt

!sudo apt-get update

In [9]:
from lab.sinus_vs_square_hard.data import create_data

nb_points_per_period = 8
nb_periods = 50
nb_points = nb_periods * nb_points_per_period
batch_size = 10 * nb_points_per_period
nb_batches = nb_points // batch_size

dataset_name = f"base_params_{nb_periods}_periods"
X, Y = create_data(nb_periods)
X = X.ravel()

In [10]:
from quantum_simulation.configs import jpc_config_dudas, quantum_parameters_dudas, base_encoding

jpc_config = jpc_config_dudas
quantum_parameters = quantum_parameters_dudas
encoding = base_encoding


In [11]:
from quantum_simulation import Simulator

simulator = Simulator(jpc_config=jpc_config,
                      encoding=encoding)


In [12]:
from quantum_learn import build_f

build_f_quad = build_f.BuildFQuadratures(jpc_config=jpc_config_dudas)
build_f_quad_poly = build_f.BuildFQuadraturesPolynomials(jpc_config=jpc_config_dudas)
build_f_probas = build_f.BuildFPhotonDistribution(jpc_config=jpc_config_dudas, clip_probas=2)


In [13]:
import numpy as np

F_quad = np.empty((nb_batches, batch_size, build_f_quad.output_dim))
F_quad_poly = np.empty((nb_batches, batch_size, build_f_quad_poly.output_dim))
F_probas = np.empty((nb_batches, batch_size, build_f_probas.output_dim))

In [ ]:
for i in range(nb_batches):
    X_batch = X[i * batch_size: (i + 1) * batch_size]
    results = simulator.run_simulation(X=X_batch, quantum_parameters_list=[quantum_parameters])[0]
    F_quad[i, :, :] = build_f_quad(results)
    F_quad_poly[i, :, :] = build_f_quad_poly(results)
    F_probas[i, :, :] = build_f_probas(results)

utzeqg



|          |   0.0% ◆ elapsed 0.00ms ◆ remaining ?
|          |   0.0% ◆ elapsed 13.49ms ◆ remaining 0.00ms
|          |   0.1% ◆ elapsed 201.38ms ◆ remaining 05m35s
|          |   0.1% ◆ elapsed 202.54ms ◆ remaining 05m35s
|          |   0.1% ◆ elapsed 383.33ms ◆ remaining 07m28s
|          |   0.1% ◆ elapsed 384.43ms ◆ remaining 07m28s
|          |   0.1% ◆ elapsed 567.24ms ◆ remaining 07m32s
|          |   0.1% ◆ elapsed 568.56ms ◆ remaining 07m32s
|          |   0.2% ◆ elapsed 747.12ms ◆ remaining 07m31s
|          |   0.2% ◆ elapsed 748.36ms ◆ remaining 07m31s
|          |   0.2% ◆ elapsed 926.80ms ◆ remaining 08m14s
|          |   0.2% ◆ elapsed 928.06ms ◆ remaining 08m14s
|          |   0.2% ◆ elapsed 1.11s ◆ remaining 07m58s   
|          |   0.2% ◆ elapsed 1.11s ◆ remaining 07m58s
|          |   0.3% ◆ elapsed 1.29s ◆ remaining 08m32s
|          |   0.3% ◆ elapsed 1.29s ◆ remaining 08m32s
|          |   0.3% ◆ elapsed 1.47s ◆ remaining 08m55s
|          |   0.3% ◆ elapsed 1.4

In [ ]:
import numpy as np
from pathlib import Path

dir = Path("datasets") / dataset_name

dir.mkdir(parents=True, exist_ok=True)

np.save(dir / "X.npy", X)
np.save(dir / "Y.npy", Y)
np.save(dir / "f_quad.npy", F_quad)
np.save(dir / "f_quad_poly.npy", F_quad_poly)
np.save(dir / "f_probas.npy", F_probas)

jpc_config_dudas.save(dir)
quantum_parameters.save(dir)
encoding.save(dir)
